In [8]:
# 1. INSTALL LIBRARIES
# !pip install transformers datasets torch scikit-learn pandas accelerate

import pandas as pd
import torch
import numpy as np
import shutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from datasets import Dataset

# --- CONFIGURATION ---
MODEL_NAME = "microsoft/codebert-base"
OUTPUT_DIR = "./edlre_final_model"

# ✅ UPDATED: Points to the merged "Syllabus + Real World" file
DATA_FILE = "neuromentor_final_training.csv"

# --- 1. LOAD & PREPARE DATA ---
print(f"📥 Loading Dataset: {DATA_FILE}...")
try:
    df = pd.read_csv(DATA_FILE)
    print(f"   ✅ Loaded {len(df)} rows.")
except FileNotFoundError:
    print(f"   ❌ ERROR: '{DATA_FILE}' not found. Please upload the merged CSV file!")
    exit()

# Map labels
label_map = { "Syntax Error": 0, "Semantic Error": 1, "Logical Error": 2, "No Error": 3 }

# Clean Data (Remove any rows with weird labels)
df = df[df['error_class'].isin(label_map.keys())]
df['label'] = df['error_class'].map(label_map)

# Split (80% Train, 20% Test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# --- 2. TOKENIZATION ---
print("🔠 Tokenizing Code (Max Length: 512)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    # ✅ UPDATED: 512 allows the model to see the full Boilerplate context
    return tokenizer(examples["code"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

# --- 3. METRICS ---
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return { 'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall }

# --- 4. INITIALIZE MODEL ---
print("🧠 Loading CodeBERT...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label={v: k for k, v in label_map.items()},
    label2id=label_map
)

# --- 5. TRAIN WITH OPTIMIZATIONS ---
print("🏋️ Starting Optimized Training...")

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,

    # ✅ MEMORY OPTIMIZATION for 512 Length
    per_device_train_batch_size=8,   # Lower batch size to save memory
    gradient_accumulation_steps=2,   # Simulate batch size 16 (8 * 2)
    per_device_eval_batch_size=8,

    num_train_epochs=10,             # Early Stopping will handle the rest
    fp16=True,                       # Fast training on GPU
    weight_decay=0.01,
    warmup_ratio=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,              # Save disk space
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Stop if not improving
)

trainer.train()

# --- 6. EVALUATE & SAVE ---
print("\n📊 Final Evaluation:")
results = trainer.evaluate()
print(f"   🏆 Accuracy: {results['eval_accuracy']:.2%}")
print(f"   📉 Loss: {results['eval_loss']:.4f}")

# Save Model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Model saved to folder: {OUTPUT_DIR}")

# Zip it for download
print("📦 Zipping model for download...")
shutil.make_archive('neuromentor_model', 'zip', OUTPUT_DIR)
print("🎉 DONE! Download 'neuromentor_model.zip' from the files tab.")

📥 Loading Dataset: neuromentor_final_training.csv...
   ✅ Loaded 4600 rows.
🔠 Tokenizing Code (Max Length: 512)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

/tmp/ipython-input-3143542502.py:101: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.022019,0.993478,0.993491,0.993547,0.993478
2,No log,0.000655,1.000000,1.000000,1.000000,1.000000
3,0.324800,0.000372,1.000000,1.000000,1.000000,1.000000
4,0.324800,0.000265,1.000000,1.000000,1.000000,1.000000
5,0.002200,0.000193,1.000000,1.000000,1.000000,1.000000



📊 Final Evaluation:


   🏆 Accuracy: 100.00%
   📉 Loss: 0.0007

✅ Model saved to folder: ./edlre_final_model
📦 Zipping model for download...
🎉 DONE! Download 'neuromentor_model.zip' from the files tab.


In [9]:
import shutil
import os
from google.colab import files

output_dir = './edlre_final_model'
zip_file_name = 'edlre_final_model.zip'

# Create a zip archive of the output directory
shutil.make_archive(output_dir, 'zip', output_dir)

# Download the zipped file
files.download(zip_file_name)

print(f"Successfully created and downloaded '{zip_file_name}'")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully created and downloaded 'edlre_final_model.zip'
